In [1]:
import os
import requests
import json
from dotenv import load_dotenv
import re

# Load environment variables from .env file
load_dotenv()

CLOUDFLARE_ACCOUNT_ID = os.getenv('CLOUDFLARE_ACCOUNT_ID')
CLOUDFLARE_API_TOKEN = os.getenv('CLOUDFLARE_API_TOKEN')

# Define the API endpoint and headers
url = f'https://api.cloudflare.com/client/v4/accounts/{CLOUDFLARE_ACCOUNT_ID}/pages/projects'
headers = {
    'Authorization': f'Bearer {CLOUDFLARE_API_TOKEN}',
    'Content-Type': 'application/json'
}

all_projects = []
has_more_pages = True
page_number = 1

while has_more_pages:
    response = requests.get(url, headers=headers, params={'page': page_number})
    
    if response.status_code == 200:
        data = response.json()
        projects = data['result']
        all_projects.extend(projects)
        
        # Check if there are more pages
        has_more_pages = data['result_info']['total_pages'] > page_number
        page_number += 1
    else:
        print(f"Failed to retrieve projects: {response.status_code} - {response.text}")
        has_more_pages = False

# Regular expression to match domains ending with 'allwomenstalk.com'
pattern = re.compile(r".*allwomenstalk\.com$")

# Filter the data
allwomenstalk_data = [
    item for item in all_projects if any(pattern.match(domain) for domain in item['domains'])
]

other_data = [
    item for item in all_projects if not any(pattern.match(domain) for domain in item['domains'])
]
# Save the projects data to a JSON file
with open('pageslist.json', 'w') as json_file:
    json.dump(allwomenstalk_data, json_file, indent=4)

with open('pagesother.json', 'w') as json_file:
    json.dump(other_data, json_file, indent=4)

print("Projects data saved to pageslist.json")


Projects data saved to pageslist.json
